# 02 · Two-stage anchor-based detector: Faster R-CNN

## Setup

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-object-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwdet import env
from hwdet.data.download import ensure_data
from hwdet.tracker import Run, status
P = env.setup(seed=42)
ensure_data(P, labels_only=False)   # True on the Mac: annotations only
status(P)                           # which runs / stages are already done

In [ ]:
# Imports, plot style, W&B login (the key is read from Kaggle / Colab secrets; without it W&B is skipped)
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from matplotlib.patches import Patch, Rectangle
from PIL import Image

from hwdet import CLASS_NAMES, NUM_CLASSES
from hwdet.config import load_config
from hwdet.data.detection_dataset import DetectionDataset, build_targets
from hwdet.data.io import frame_table
from hwdet.engine import Data, load_weights, run_experiment
from hwdet.eval import errors
from hwdet.tracker import Run, load_registry
from hwdet.viz import COLORS, INK_3, PALETTE, save_fig, set_style
from hwdet.wandb_utils import login_from_secrets

set_style()
login_from_secrets()
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", env.device())

## 1. Data for all experiments
Shared protocol (`configs/experiments/_protocol.yaml`): every 10th training frame, the full official val
split for final metrics, and every 5th val frame to score the model after each epoch.

In [ ]:
# Frames and targets (COCO annotations -> torchvision boxes/labels + ignored regions), built once
protocol = load_config("frcnn_baseline")["split"]
train_frames = frame_table(P.data, "train")
train_frames = train_frames[(train_frames.frame - 1) % protocol["train_frame_step"] == 0]
val_frames = frame_table(P.data, "val")
monitor_frames = val_frames[(val_frames.frame - 1) % protocol["monitor_frame_step"] == 0]
test_frames = frame_table(P.data, "test")
targets = build_targets(P.data, pd.concat([train_frames, val_frames]))
data = Data(P.data, train_frames, val_frames, monitor_frames, targets, test_frames)
pd.DataFrame({"frames": [len(train_frames), len(val_frames), len(monitor_frames), len(test_frames)],
              "clips": [d["clip"].nunique() for d in (train_frames, val_frames, monitor_frames, test_frames)]},
             index=["train (subsampled)", "val", "val monitor", "test"])

Converted training samples as the model receives them: ignored regions painted with the mean colour
(boxes inside them dropped) and random horizontal flip.

In [ ]:
# Visual check of the conversion: training dataset samples with their target boxes
train_ds_view = DetectionDataset(train_frames, P.data, targets, train=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, i in zip(axes, [100, 1500]):
    img, tgt, _ = train_ds_view[i]
    ax.imshow(img.permute(1, 2, 0).numpy())
    for (x1, y1, x2, y2), l in zip(tgt["boxes"].numpy(), tgt["labels"].numpy()):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=COLORS[l - 1], lw=1.2))
    ax.set_title(f"{train_frames.file_name.iloc[i]}  ·  {len(tgt['boxes'])} boxes", fontsize=9, fontweight="normal")
    ax.axis("off")
fig.legend(handles=[Patch(fill=False, ec=COLORS[k], lw=2, label=n) for k, n in enumerate(CLASS_NAMES)],
           loc="lower center", ncol=8)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save_fig(fig, "frcnn_train_samples", P)

## 2. Model
torchvision Faster R-CNN with a ResNet-50 + FPN backbone, pretrained on COCO. The COCO class head is
replaced with a new 9-way head (8 classes + background). All settings come from the YAML config.

In [ ]:
# Builds Faster R-CNN from a config: COCO weights, new 8-class head, anchors, RPN / RoI settings, box loss
from torchvision.models.detection import (FasterRCNN_ResNet50_FPN_V2_Weights, FasterRCNN_ResNet50_FPN_Weights,
                                          fasterrcnn_resnet50_fpn, fasterrcnn_resnet50_fpn_v2)
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import RPNHead

ARCHS = {"resnet50_fpn": (fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights.COCO_V1, 1),
         "resnet50_fpn_v2": (fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, 2)}
MODEL_KWARGS = ["min_size", "max_size", "rpn_pre_nms_top_n_train", "rpn_post_nms_top_n_train",
                "rpn_pre_nms_top_n_test", "rpn_post_nms_top_n_test", "box_batch_size_per_image",
                "box_positive_fraction", "box_detections_per_img", "box_score_thresh"]
BOX_LOSS = {"name": "smooth_l1", "giou_weight": 10.0}     # read by the patched second-stage loss below


def build_frcnn(cfg):
    m = cfg["model"]
    ctor, weights, rpn_conv_depth = ARCHS[m["arch"]]
    model = ctor(weights=weights, **{k: m[k] for k in MODEL_KWARGS if k in m})
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)
    sizes = tuple((s,) for s in m["anchor_sizes"])
    model.rpn.anchor_generator = AnchorGenerator(sizes, (tuple(m["aspect_ratios"]),) * len(sizes))
    if len(m["aspect_ratios"]) != model.rpn.head.cls_logits.out_channels:   # new anchor count -> new RPN head
        model.rpn.head = RPNHead(256, len(m["aspect_ratios"]), conv_depth=rpn_conv_depth)
    BOX_LOSS.update(name=m.get("box_loss", "smooth_l1"), giou_weight=m.get("giou_weight", 10.0))
    return model

**Box loss experiment.** torchvision trains the second-stage box regression with smooth L1 on the
encoded offsets (dx, dy, dw, dh). The GIoU variant decodes predicted and target offsets relative to the
same unit box and applies the GIoU loss. IoU and GIoU do not change under a shared shift and per-axis
scaling, so this equals the GIoU of the real boxes. Weight 10 follows MMDetection's Faster R-CNN GIoU config.

In [ ]:
# Patch torchvision's second-stage loss: smooth L1 (default) or GIoU, chosen by BOX_LOSS["name"]
from torchvision.models.detection import roi_heads as rh
from torchvision.ops import generalized_box_iou_loss

_smooth_l1_loss = getattr(rh, "_hw_original_fastrcnn_loss", rh.fastrcnn_loss)
rh._hw_original_fastrcnn_loss = _smooth_l1_loss
CLAMP = math.log(1000.0 / 16)


def _unit_boxes(d, w=(10.0, 10.0, 5.0, 5.0)):
    """Offsets (dx, dy, dw, dh) -> xyxy box relative to a unit box centred at 0 (torchvision's encoding)."""
    cx, cy = d[:, 0] / w[0], d[:, 1] / w[1]
    bw, bh = torch.exp((d[:, 2] / w[2]).clamp(max=CLAMP)), torch.exp((d[:, 3] / w[3]).clamp(max=CLAMP))
    return torch.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1)


def fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    if BOX_LOSS["name"] != "giou":
        return _smooth_l1_loss(class_logits, box_regression, labels, regression_targets)
    labels, regression_targets = torch.cat(labels), torch.cat(regression_targets)
    cls_loss = F.cross_entropy(class_logits, labels)
    pos = torch.where(labels > 0)[0]
    box_regression = box_regression.reshape(class_logits.shape[0], -1, 4)
    pred = _unit_boxes(box_regression[pos, labels[pos]].float())
    tgt = _unit_boxes(regression_targets[pos].float())
    box_loss = BOX_LOSS["giou_weight"] * generalized_box_iou_loss(pred, tgt, reduction="sum") / labels.numel()
    return cls_loss, box_loss


rh.fastrcnn_loss = fastrcnn_loss

## 3. Smoke test
One short epoch on a few frames under a separate run id, to catch errors before the long runs. It is
deleted afterwards.

In [ ]:
# Smoke test: 1 epoch on ~40 frames, then the run folder and its registry row are removed
import shutil
smoke = Data(P.data, train_frames.iloc[::60], val_frames.iloc[::150], val_frames.iloc[::150], targets)
run_experiment("frcnn_baseline", build_frcnn, P, smoke, run_id="smoke_frcnn", wandb=False,
               train={"epochs": 1, "warmup_iters": 5})
shutil.rmtree(P.runs / "smoke_frcnn", ignore_errors=True)
reg_file = P.outputs / "experiments.csv"
if reg_file.exists():
    reg = pd.read_csv(reg_file)
    reg[reg.run_id != "smoke_frcnn"].to_csv(reg_file, index=False)

## 4. Experiments
Each cell trains one config (≈1.5 h on a Kaggle P100 / Colab T4), evaluates the best epoch on the full
val split and saves everything under `outputs/runs/<config>/`. Re-running a cell after a crash continues
where it stopped.

| config | change vs baseline |
|---|---|
| `frcnn_baseline` | torchvision defaults, 300 detections per image |
| `frcnn_small_anchors` | anchors 16–256 px instead of 32–512, 2× RPN proposals |
| `frcnn_roi_sampling` | 1024 RoIs per image, 50 % positives (default 512, 25 %) |
| `frcnn_giou` | GIoU loss for the second-stage box regression |

In [ ]:
# Baseline
runs = {}
runs["frcnn_baseline"] = run_experiment("frcnn_baseline", build_frcnn, P, data)

In [ ]:
# Experiment 1: smaller anchors + more RPN proposals
runs["frcnn_small_anchors"] = run_experiment("frcnn_small_anchors", build_frcnn, P, data)

In [ ]:
# Experiment 2: RoI sampling for the second stage
runs["frcnn_roi_sampling"] = run_experiment("frcnn_roi_sampling", build_frcnn, P, data)

In [ ]:
# Experiment 3: GIoU box regression loss
runs["frcnn_giou"] = run_experiment("frcnn_giou", build_frcnn, P, data)

## 5. Results

In [ ]:
# Validation metrics of all Faster R-CNN runs (full val split, best epoch of each run)
reg = load_registry(P)
cols = ["run_id", "description", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms_per_img"]
results = reg[(reg.family == "two_stage") & (reg["eval"] == "val")][cols].sort_values("mAP", ascending=False)
results.round(4)

In [ ]:
# Per-class AP of all runs
per_class = reg[(reg.family == "two_stage") & (reg["eval"] == "val")].set_index("run_id")
(per_class[[f"AP_{c}" for c in CLASS_NAMES]].rename(columns=lambda c: c[3:]) * 100).round(1)

In [ ]:
# Training curves: total loss and val mAP (monitor subset) per epoch for every run
hist = {r: Run(r, None, P, use_wandb=False).history for r in results.run_id}
fig, (a0, a1) = plt.subplots(1, 2, figsize=(14, 4.2))
for (name, h), col in zip(hist.items(), PALETTE):
    loss_cols = [c for c in h.columns if c.startswith("loss_")]
    a0.plot(h.epoch, h[loss_cols].sum(1), marker="o", color=col, label=name)
    a1.plot(h.epoch, h.val_mAP, marker="o", color=col, label=name)
a0.set(title="Training loss (sum of the four losses)", xlabel="epoch")
a1.set(title="Val mAP (monitor subset)", xlabel="epoch")
a1.legend(loc="lower right")
fig.tight_layout()
save_fig(fig, "frcnn_curves", P)

## 6. What each stage contributes
The RPN proposes class-agnostic candidate boxes; the second stage classifies each proposal into one of
8 classes or background and refines its box. Recall of the proposals (any box with IoU ≥ 0.5 with an
object) is compared with recall of the final detections, by object size.

In [ ]:
# Best run by val mAP, reloaded from its best checkpoint
best_id = results.run_id.iloc[0]
best_cfg = load_config(best_id)
model = load_weights(build_frcnn(best_cfg), Run(best_id, None, P, use_wandb=False), "best", env.device())
dev = env.device()
print("best run:", best_id)

In [ ]:
# Recall of RPN proposals (top-k) vs final detections on the monitor frames, per object size
from torchvision.ops import box_iou

val_ds = DetectionDataset(monitor_frames, P.data, targets)
SIZES = {"small": (0, 32 ** 2), "medium": (32 ** 2, 96 ** 2), "large": (96 ** 2, 1e12)}
hits = {k: [] for k in ["top 100 proposals", "top 300 proposals", "all proposals", "final detections"]}
areas = []
model.eval()
with torch.no_grad():
    for i in range(0, len(val_ds), 2):
        img, tgt, _ = val_ds[i]
        gt = tgt["boxes"].to(dev)
        if len(gt) == 0:
            continue
        images, _ = model.transform([img.to(dev)])
        feats = model.backbone(images.tensors)
        props, _ = model.rpn(images, feats)
        h, w = images.image_sizes[0]
        props = props[0] * torch.tensor([img.shape[2] / w, img.shape[1] / h] * 2, device=dev)
        det = model([img.to(dev)])[0]
        det_boxes = det["boxes"][det["scores"] >= 0.05]
        for name, boxes in [("top 100 proposals", props[:100]), ("top 300 proposals", props[:300]),
                            ("all proposals", props), ("final detections", det_boxes)]:
            hits[name].append((box_iou(gt, boxes).max(1).values >= 0.5).cpu().numpy() if len(boxes)
                              else np.zeros(len(gt), bool))
        areas.append(((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])).cpu().numpy())
areas = np.concatenate(areas)
stage_recall = pd.DataFrame({name: {s: np.concatenate(v)[(areas >= lo) & (areas < hi)].mean()
                                    for s, (lo, hi) in SIZES.items()} | {"all": np.concatenate(v).mean()}
                             for name, v in hits.items()}).T
stage_recall.round(3)

In [ ]:
# One frame: GT, top-100 proposals (class-agnostic) and final detections (score >= 0.5)
i = len(val_ds) // 2
img, tgt, _ = val_ds[i]
with torch.no_grad():
    images, _ = model.transform([img.to(dev)])
    props, _ = model.rpn(images, model.backbone(images.tensors))
    h, w = images.image_sizes[0]
    props = (props[0] * torch.tensor([img.shape[2] / w, img.shape[1] / h] * 2, device=dev)).cpu().numpy()
    det = model([img.to(dev)])[0]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
panels = [("ground truth", tgt["boxes"].numpy(), tgt["labels"].numpy() - 1),
          ("RPN: top-100 proposals", props[:100], None),
          ("final detections, score ≥ 0.5", det["boxes"][det["scores"] >= 0.5].cpu().numpy(),
           det["labels"][det["scores"] >= 0.5].cpu().numpy() - 1)]
for ax, (title, boxes, cls) in zip(axes, panels):
    ax.imshow(img.permute(1, 2, 0).numpy())
    for k, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1,
                               ec=INK_3 if cls is None else COLORS[cls[k]]))
    ax.set_title(f"{title} ({len(boxes)})", fontsize=10, fontweight="normal")
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "frcnn_stages", P)

## 7. Failure cases
Detections of the best run on the monitor frames are matched to the ground truth (`hwdet/eval/errors.py`)
at the score threshold with the best F1, and every error gets a type.

In [ ]:
# Match val predictions to ground truth, choose the best-F1 score threshold, summarise error types
preds = pd.read_csv(Run(best_id, None, P, use_wandb=False).pred_path("val"))
dets, gts = errors.match(preds, monitor_frames, targets)
thr, f1 = errors.best_f1_threshold(dets, gts)
summ = errors.summary(dets, gts, thr)
print(f"best-F1 threshold {thr} (F1 {f1:.3f})")
display(summ["outcomes"].T)
display(summ["recall_by_size"].round(3).T)
summ["recall_by_class"].round(3).T

In [ ]:
# The 3 frames with the most errors: GT (dashed) vs detections coloured by outcome
OUT_COL = {"correct": PALETTE[2], "duplicate": PALETTE[3], "wrong_class": PALETTE[6],
           "poor_localisation": PALETTE[0], "background": PALETTE[7], "ignored": INK_3}
worst = errors.worst_frames(dets, gts, thr, n=3)
display(worst)
fig, axes = plt.subplots(len(worst), 1, figsize=(14, 7.5 * len(worst)))
for ax, fn in zip(np.atleast_1d(axes), worst.index):
    ax.imshow(Image.open(P.data / fn))
    t = targets[fn]
    for x1, y1, x2, y2 in t["boxes"]:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="white", ls="--", lw=1))
    for b in dets[(dets.file_name == fn) & (dets.score >= thr)].itertuples():
        ax.add_patch(Rectangle((b.x, b.y), b.w, b.h, fill=False, ec=OUT_COL[b.outcome], lw=1.4))
    ax.set_title(f"{fn}  ·  missed {worst.loc[fn, 'missed']}, false positives {worst.loc[fn, 'false_pos']}",
                 fontsize=10, fontweight="normal")
    ax.axis("off")
fig.legend(handles=[Patch(fill=False, ec=c, lw=2, label=k) for k, c in OUT_COL.items()]
           + [Patch(fill=False, ec="grey", ls="--", label="ground truth")], loc="lower center", ncol=7)
fig.tight_layout(rect=(0, 0.02, 1, 1))
save_fig(fig, "frcnn_failures", P)

## 8. Test predictions of the best run

In [ ]:
# Test-set predictions + Kaggle submission for the best run (training and val stages are skipped)
best_run = run_experiment(best_id, build_frcnn, P, data, predict_test=True)
print(best_run.dir / "submission.csv")